<a href="https://colab.research.google.com/github/Dias21B030874/kazakh_profanity_detection/blob/main/KazakhProfanityDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from datasets import load_dataset

full = load_dataset("issai/kazsandra", "full")
pc = load_dataset("issai/kazsandra", "polarity_classification")
sc = load_dataset("issai/kazsandra", "score_classification")

full/full.csv: reconstructing file:   0%|          |  0.00B / 24.4MB            

full/full.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/180064 [00:00<?, ? examples/s]

polarity_classification/train_pc.csv: reconstructing file:   0%|          |  0.00B / 32.6MB            

polarity_classification/train_pc.csv: downloading bytes:           |  0.00B            

valid_pc.csv:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

... (7 outputs left out)

test_sc.csv:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/140126 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17516 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17516 [00:00<?, ? examples/s]

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [6]:
from datasets import load_dataset

ds = load_dataset("Rustem-Kaimolla/kazakh-swear-words")

README.md:   0%|          | 0.00/3.17k [00:00<?, ?B/s]

data.jsonl:   0%|          | 0.00/5.46k [00:00<?, ?B/s]

sentences.jsonl:   0%|          | 0.00/7.19k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:
import pandas as pd
import re

df_reviews = pd.DataFrame(full['train'])
df_swear = pd.DataFrame(ds['train'])

print("--- KazSAnDRA ---")
print("Size:", df_reviews.shape)
print("Columns:", df_reviews.columns.tolist())
display(df_reviews.head(2))

print("\n--- Словарь мата ---")
print("Size:", df_swear.shape)
print("Columns:", df_swear.columns.tolist())
display(df_swear.head(2))

--- KazSAnDRA ---
Size: (180064, 4)
Columns: ['custom_id', 'text', 'label', 'domain']


,custom_id,text,label,domain
0,gis000001gis,Тамақтарыңыз өте дәмді екен. Көп көп рақмет,5,mapping
1,gis000002gis,"Пасуда кір , официанттардын устінен сасык иіс ...",1,mapping



--- Словарь мата ---
Size: (119, 5)
Columns: ['text', 'label', 'category', 'language', 'severity']


,text,label,category,language,severity
0,қотақ,1,vulgar,kk,severe
1,тасақ,1,vulgar,kk,severe


In [10]:
import pandas as pd
import re

# 1. Готовим словарь мата
# Извлекаем слова, переводим в нижний регистр, убираем пробелы
swear_words = df_swear['text'].dropna().str.lower().str.strip().tolist()
swear_words = [w for w in swear_words if len(w) >= 3]

# Создаем регулярное выражение для поиска
pattern = r'\b(?:' + '|'.join(map(re.escape, swear_words)) + r')'

# 2. Ищем мат в большом датасете KazSAnDRA
# Удаляем пустые тексты на всякий случай
df_reviews = df_reviews.dropna(subset=['text'])

df_reviews['is_profane'] = df_reviews['text'].str.lower().str.contains(pattern, regex=True).astype(int)

profane_count = df_reviews['is_profane'].sum()
print(f"Отзывов с потенциальным матом найдено: {profane_count}\n")

# 3. Собираем сбалансированный датасет (примерно 3000-4000 строк)
df_profane = df_reviews[df_reviews['is_profane'] == 1].sample(n=min(1500, profane_count), random_state=42)
target_count = len(df_profane)

# Берем СТОЛЬКО ЖЕ обычных негативных отзывов (рейтинг 1 или 2), но БЕЗ мата
df_clean_negative = df_reviews[(df_reviews['is_profane'] == 0) & (df_reviews['label'].astype(str).isin(['1', '2', '1.0', '2.0']))].sample(n=target_count, random_state=42)

# Берем половину от этого количества позитивных отзывов (рейтинг 4 или 5)
df_clean_positive = df_reviews[(df_reviews['is_profane'] == 0) & (df_reviews['label'].astype(str).isin(['4', '5', '4.0', '5.0']))].sample(n=target_count // 2, random_state=42)

# 4. Объединяем, перемешиваем и оставляем только нужные колонки
df_final = pd.concat([df_profane, df_clean_negative, df_clean_positive]).sample(frac=1, random_state=42).reset_index(drop=True)
df_final = df_final[['text', 'is_profane']] # Оставляем только текст и наш таргет

print(f"Итоговый размер собранного датасета: {len(df_final)}")
print("\nРаспределение классов (1 - мат, 0 - чисто):")
print(df_final['is_profane'].value_counts())

# Выводим несколько примеров с матом для ручной проверки
print("\nПримеры найденного мата:")
display(df_final[df_final['is_profane'] == 1].head(5))

Отзывов с потенциальным матом найдено: 62046

Итоговый размер собранного датасета: 3750

Распределение классов (1 - мат, 0 - чисто):
is_profane
0    2250
1    1500
Name: count, dtype: int64

Примеры найденного мата:


,text,is_profane
1,Балаларға өте ыңғайлы.,1
3,Күшті осы онлайн мектепке рахмет,1
6,"Өте жақсы, сапасы да керемет, суды тез жылытады.",1
7,Рахмет. Алла разы болсын. Маган катты унады.,1
9,Өте жақсы бағдарлама 5 қой!!!!,1


In [11]:
df_reviews['matched_swear'] = df_reviews['text'].str.lower().str.findall(pattern)

pd.set_option('display.max_colwidth', None)
display(df_reviews[df_reviews['is_profane'] == 1][['text', 'matched_swear']].head(10))

all_matches = df_reviews['matched_swear'].dropna().explode()
print("\nТоп-10 слов из словаря, которые срабатывали чаще всего:")
print(all_matches.value_counts().head(10))

,text,matched_swear
0,Тамақтарыңыз өте дәмді екен. Көп көп рақмет,[тамақ]
3,"Түскі 12де келдік. Е анау жоқ, е мынау жоқ. Не бар сонда, адамдар тамақ ішуге келеді ғой, тамақ жоқ!","[ана, адам, тамақ, тамақ]"
4,"Масқара! Кеше ауызашарда масқара болдық, 15200 теңгеге нан ішіндегі қуырдаққа тапсырыс беріп едік. ішінде болуы керек қоспалар жоқ, ет 1 келіге де жетпейді, кортобы да аз, сәбізін көрмедік, өте нашар! 6 адам жеу керек тағамды 2 адам жеп тоймай қалды. Қонақтардан ұялғанымызай. Тапсырыс берерде мұқият болыңыздар!!!!","[адам, адам]"
6,"Өте нашар қызмет көрсету, тамақтары тым кеш келеді +суық күйінде келеді. Столдың үстері таза емес дұрыс сүртілмейді",[тамақ]
7,"Қызмет көрсету өте өте нашар. Адам таңдап қызмет көрсетеды, айтылған затты уақытында әкелмейді",[адам]
8,"Қызмет көрсетуі нашар! Бірақ, тағамдары жақсы.",[жақсы]
12,"Сәлеметсіз бе. 6.01.23 тойымыз осында өтті. Өте ұнады, рақмет. Еңбектеріңіз жемісті болсын!!! Іші әдемі, тамақтары тәтті, администратор мен официанттары пысық.","[сәлем, әдемі, тамақ]"
13,"27.12.22ж күні Жаңа жылдық корпоративте болдық. Банкет залы керемет ұнады. Тағамдары да, услугалары да ұнады. Администраторларына, қызметкерлеріне рахмет айтамын","[күн, рахмет]"
14,"Мейрамхананың администраторына, аспазшыларына көп рахмет! Еңбектеріңіздің жемісін көріңіздер",[рахмет]
15,Өте жақсы күшті,[жақсы]



Топ-10 слов из словаря, которые срабатывали чаще всего:
matched_swear
жақсы     31247
рахмет    12991
қала       5932
адам       5470
жұмыс      3878
жер        3796
бала       3758
күн        3219
әдемі      2955
жұлдыз     2904
Name: count, dtype: int64


In [12]:
all_swear_words = df_swear['text'].dropna().str.lower().str.strip().sort_values().tolist()

print(f"Всего слов в словаре: {len(all_swear_words)}\n")
print(all_swear_words)

Всего слов в словаре: 119

['667', 'адам', 'ай', 'акеннын аузын сыгем', 'ам', 'амбас', 'амшық', 'амщелек', 'амыңды шығарам', 'ана', 'арсыз', 'аспан', 'ата', 'ауыл', 'ақымақ', 'бала', 'барлығы жақсы болады', 'бақытты болыңыз', 'бок же', 'бүгін ауа райы жақсы', 'бұл тамаша идея', 'гүл', 'денсаулығыңызға', 'долбоебсын ба', 'дос', 'достық', 'ертең көріскенше', 'есер', 'жарайсың', 'жақсы', 'жақсы демалыңыз', 'жер', 'жынды', 'жұлдыз', 'жұмыс', 'жәлеп', 'итпұшық', 'каншык', 'кет нахуй', 'кет нахүй', 'кешіріңіз, түсінбедім', 'котагы ма', 'котағы ма', 'котинды айрам', 'котинды кыс', 'котын кышып тур ма', 'котын сыгейн', 'кітап', 'күлімдеңіз', 'күн', 'көмектесе аласыз ба', 'көтің сігейін', 'көтің қышып тұр ма', 'көтіңді айырамын', 'көтіңді қыс', 'көше', 'маган поxyй', 'махаббат', 'маған поxyй', 'мектеп', 'мен жақсымын, рахмет', 'мен сізге көмектесемін', 'мен сізді жақсы көремін', 'мен үйге барамын', 'отбасы', 'оттама', 'пұшық', 'рахмет', 'рахмет сізге', 'салдакы', 'салдақы', 'сасық', 'су', 'сыкт

In [18]:
import pandas as pd
import re

swear_words = df_swear[df_swear['label'] == 1]['text'].dropna().str.lower().str.strip().tolist()

base_kz_patterns = [r'\b' + w + r'\w*' for w in set(swear_words)]

kz_masked_patterns = [
    r'\bқ[оө\*\.\-]+т[а\*\.\-]+қ\w*',     # қотақ
    r'\bш[е\*\.\-]+ш[е\*\.\-]+[ңн]\w*',   # шешең / шешен
    r'\bк[өо\*\.\-]+т[іи\*\.\-]+[ңн]\w*', # көтің / көтін
    r'\bс[іи\*\.\-]+г[е\*\.\-]+[м\w]\w*', # сігем / сігіл
    r'\bа[м\*\.\-]+ш[ыи\*\.\-]+қ\w*'      # амшық
]

ru_and_masked_patterns = [
    r'\bб[л\*\.]+я\w*', r'\bп[ие\*\.]+зд\w*',
    r'\bе[б\*\.]+а\w*', r'\bх[у\*\.]+й\w*'
]

all_patterns = base_kz_patterns + kz_masked_patterns + ru_and_masked_patterns
full_profanity_pattern = r'(?:' + '|'.join(all_patterns) + r')'

df_reviews = df_reviews.dropna(subset=['text'])
df_reviews['is_profane'] = df_reviews['text'].str.lower().str.contains(full_profanity_pattern, regex=True).astype(int)

profane_count = df_reviews['is_profane'].sum()
print(f"Отзывов с матом (включая замаскированный) найдено: {profane_count}")

df_profane = df_reviews[df_reviews['is_profane'] == 1].copy()
target_count = len(df_profane)

if target_count > 0:
    df_clean_negative = df_reviews[
        (df_reviews['is_profane'] == 0) &
        (df_reviews['label'].astype(str).isin(['1', '2', '1.0', '2.0']))
    ].sample(n=target_count, random_state=42)

    df_clean_positive = df_reviews[
        (df_reviews['is_profane'] == 0) &
        (df_reviews['label'].astype(str).isin(['4', '5', '4.0', '5.0']))
    ].sample(n=target_count, random_state=42)

    df_final = pd.concat([df_profane, df_clean_negative, df_clean_positive]).sample(frac=1, random_state=42).reset_index(drop=True)
    df_final = df_final[['text', 'is_profane']]

    print(f"\nИтоговый размер собранного датасета: {len(df_final)}")
    print(df_final['is_profane'].value_counts())

    df_final.to_csv("kazakh_profanity_dataset.csv", index=False)
    print("\nФайл 'kazakh_profanity_dataset.csv' успешно сохранен!")
else:
    print("Мат не найден, проверьте датасет.")

Отзывов с матом (включая замаскированный) найдено: 2701

Итоговый размер собранного датасета: 8103
is_profane
0    5402
1    2701
Name: count, dtype: int64

Файл 'kazakh_profanity_dataset.csv' успешно сохранен!


In [15]:
swear_words = df_swear[df_swear['label'] == 1]['text'].dropna().str.lower().str.strip().tolist()
safe_swear_words = [w for w in swear_words if w != 'ам']

# Базовые казахские корни с любыми окончаниями (кроме "ам")
base_kz_patterns = [r'\b' + w + r'\w*' for w in set(safe_swear_words)]

# Прописываем только реальные матерные формы, чтобы не ловить "амалы", "аман"
dangerous_roots = [
    r'\bам\b', r'\bамы\b', r'\bамын\b', r'\bамың\b', r'\bамыңды\b'
]

# КАЗАХСКИЙ МАТ
kz_masked_patterns = [
    r'\bқ[оө\*\.\-]+т[а\*\.\-]+қ\w*',
    r'\bш[е\*\.\-]+ш[е\*\.\-]+[ңн]\w*',
    r'\bк[өо\*\.\-]+т[іи\*\.\-]+[ңн]\w*',
    r'\bс[іи\*\.\-]+г[е\*\.\-]+[м\w]\w*',
    r'\bа[м\*\.\-]+ш[ыи\*\.\-]+қ\w*'
]

# РУССКИЙ МАТ
ru_and_masked_patterns = [
    r'\bб[л\*\.]+я\w*', r'\bп[ие\*\.]+зд\w*',
    r'\bе[б\*\.]+а\w*', r'\bх[у\*\.]+й\w*'
]

all_patterns = base_kz_patterns + dangerous_roots + kz_masked_patterns + ru_and_masked_patterns
full_profanity_pattern = r'(?:' + '|'.join(all_patterns) + r')'

In [17]:
df_reviews = df_reviews.dropna(subset=['text'])
df_reviews['is_profane'] = df_reviews['text'].str.lower().str.contains(full_profanity_pattern, regex=True).astype(int)

profane_count = df_reviews['is_profane'].sum()
print(f"Отзывов с матом (включая замаскированный) найдено: {profane_count}")

df_profane = df_reviews[df_reviews['is_profane'] == 1].copy()
target_count = len(df_profane)

if target_count > 0:
    df_clean_negative = df_reviews[
        (df_reviews['is_profane'] == 0) &
        (df_reviews['label'].astype(str).isin(['1', '2', '1.0', '2.0']))
    ].sample(n=target_count, random_state=42)

    df_clean_positive = df_reviews[
        (df_reviews['is_profane'] == 0) &
        (df_reviews['label'].astype(str).isin(['4', '5', '4.0', '5.0']))
    ].sample(n=target_count, random_state=42)

    df_final = pd.concat([df_profane, df_clean_negative, df_clean_positive]).sample(frac=1, random_state=42).reset_index(drop=True)
    df_final = df_final[['text', 'is_profane']]

    print(f"\nИтоговый размер собранного датасета: {len(df_final)}")
    print(df_final['is_profane'].value_counts())

    df_final.to_csv("kazakh_profanity_dataset2.csv", index=False)
    print("\nФайл 'kazakh_profanity_dataset2.csv' успешно сохранен!")
else:
    print("Мат не найден, проверьте датасет.")

Отзывов с матом (включая замаскированный) найдено: 1947

Итоговый размер собранного датасета: 5841
is_profane
0    3894
1    1947
Name: count, dtype: int64

Файл 'kazakh_profanity_dataset2.csv' успешно сохранен!


In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib

df = pd.read_csv("kazakh_profanity_dataset2.csv")
df = df.dropna(subset=['text'])

# Отделяем 15% на финальный ТЕСТ
X_temp, X_test, y_temp, y_test = train_test_split(
    df['text'], df['is_profane'],
    test_size=0.15, random_state=42, stratify=df['is_profane']
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176, random_state=42, stratify=y_temp
)

print(f"Размер обучающей выборки (Train): {len(X_train)}")
print(f"Размер валидационной выборки (Val): {len(X_val)}")
print(f"Размер тестовой выборки (Test): {len(X_test)}\n")

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), max_features=15000)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model.fit(X_train_vec, y_train)


val_probs = model.predict_proba(X_val_vec)[:, 1]
custom_threshold = 0.60
y_val_pred_custom = (val_probs >= custom_threshold).astype(int)

test_probs = model.predict_proba(X_test_vec)[:, 1]
y_test_pred = (test_probs >= custom_threshold).astype(int)

print("=== ФИНАЛЬНЫЕ МЕТРИКИ НА TEST SET ===")
print(classification_report(y_test, y_test_pred, target_names=['Clean (0)', 'Profane (1)']))

print("\n=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_test, y_test_pred)
print(f"True Negatives (Верно чистые): {cm[0][0]}")
print(f"False Positives (Ошибка: ложный мат): {cm[0][1]}")
print(f"False Negatives (Ошибка: пропуск мата): {cm[1][0]}")
print(f"True Positives (Верно мат): {cm[1][1]}")

# Сохранение весов
joblib.dump(model, 'kazakh_profanity_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

Размер обучающей выборки (Train): 4090
Размер валидационной выборки (Val): 874
Размер тестовой выборки (Test): 877

=== ФИНАЛЬНЫЕ МЕТРИКИ НА TEST SET ===
              precision    recall  f1-score   support

   Clean (0)       0.93      0.99      0.96       585
 Profane (1)       0.98      0.84      0.91       292

    accuracy                           0.94       877
   macro avg       0.96      0.92      0.93       877
weighted avg       0.95      0.94      0.94       877


=== CONFUSION MATRIX ===
True Negatives (Верно чистые): 581
False Positives (Ошибка: ложный мат): 4
False Negatives (Ошибка: пропуск мата): 46
True Positives (Верно мат): 246


['tfidf_vectorizer.pkl']